**ФИО студента:** [Иванов Иван Иванович]

**Задание:** [выберите между `inpainting` / `outpainting`]

**Группа:** [номер группы]

**Дата сдачи:** [дата]

## Блок настройки окружения

Код для импорта библиотек, для формирования глобальных констант, других конфигураций (если требуется). Постарайтесь здесь зафиксировать все seed, которые сможете

In [ ]:
...

## Блок "Данные"

Код и информация о том, как были подготовлены данные для обучения модели

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


cube = np.load("input_seismic_data/inpainting_cube_missed_data.npy").astype(np.float32)
real_mask = np.zeros(cube.shape, dtype=bool)
real_mask[:, 368:581, 179:282] = True

PATCH_H = 320
PATCH_W = 160
MASK_H = 213
MASK_W = 103
REGIONS = (
    (0, 368, 0, 462),
    (581, 951, 0, 462),
    (0, 951, 0, 179),
    (0, 951, 282, 462)
)

sample = cube[::4, ::4, ::4]
sample_known = np.isfinite(sample) & ~real_mask[::4, ::4, ::4]
mean = float(sample[sample_known].mean())
std = max(float(sample[sample_known].std()), 1e-6)


def make_samples(centers, samples_per_center, augment, seed):
    rng = np.random.default_rng(seed)
    samples = []

    for center in centers:
        for _ in range(samples_per_center):
            y_min, y_max, z_min, z_max = REGIONS[rng.integers(len(REGIONS))]
            y = int(rng.integers(y_min, y_max - PATCH_H + 1))
            z = int(rng.integers(z_min, z_max - PATCH_W + 1))
            mask_y = int(rng.integers(16, PATCH_H - MASK_H - 15))
            mask_z = int(rng.integers(16, PATCH_W - MASK_W - 15))
            flip_h = augment and bool(rng.integers(2))
            flip_v = augment and bool(rng.integers(2))
            samples.append((center, y, z, mask_y, mask_z, flip_h, flip_v))

    return samples


class InpaintingDataset(Dataset):
    def __init__(self, cube, samples, mean, std):
        self.cube = cube
        self.samples = samples
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        center, y, z, mask_y, mask_z, flip_h, flip_v = self.samples[index]
        patch = self.cube[
            center - 2:center + 3,
            y:y + PATCH_H,
            z:z + PATCH_W
        ].copy()
        target = patch[2].copy()

        patch = (patch - self.mean) / self.std
        target = (target - self.mean) / self.std

        mask = np.zeros((PATCH_H, PATCH_W), dtype=np.float32)
        mask[mask_y:mask_y + MASK_H, mask_z:mask_z + MASK_W] = 1
        patch *= (1 - mask)[None]

        if flip_h:
            patch = patch[:, :, ::-1]
            target = target[:, ::-1]
            mask = mask[:, ::-1]

        if flip_v:
            patch = patch[:, ::-1, :]
            target = target[::-1, :]
            mask = mask[::-1, :]

        inputs = np.concatenate((patch, mask[None]), axis=0)

        return (
            torch.from_numpy(inputs.copy()).float(),
            torch.from_numpy(target[None].copy()).float(),
            torch.from_numpy(mask[None].copy()).float()
        )


train_centers = np.arange(2, 402)
val_centers = np.arange(406, 492)
test_centers = np.arange(496, 578)
train_samples = make_samples(train_centers, 8, True, 42)
val_samples = make_samples(val_centers, 4, False, 43)
test_samples = make_samples(test_centers, 4, False, 44)

train_dataset = InpaintingDataset(cube, train_samples, mean, std)
val_dataset = InpaintingDataset(cube, val_samples, mean, std)
test_dataset = InpaintingDataset(cube, test_samples, mean, std)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


## Блок "Модель"

Код и информация о том, какая модель была использована

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class UNet25D(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.encoders = nn.ModuleList([
            ConvBlock(in_channels, base_channels),
            ConvBlock(base_channels, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 8)
        ])
        self.bottleneck = ConvBlock(base_channels * 8, base_channels * 16)
        self.upsamplers = nn.ModuleList([
            nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, stride=2),
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        ])
        self.decoders = nn.ModuleList([
            ConvBlock(base_channels * 16, base_channels * 8),
            ConvBlock(base_channels * 8, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels)
        ])
        self.head = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_height = (-height) % 16
        pad_width = (-width) % 16
        x = F.pad(x, (0, pad_width, 0, pad_height))

        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for upsampler, decoder, skip in zip(
            self.upsamplers,
            self.decoders,
            reversed(skips)
        ):
            x = upsampler(x)
            x = torch.cat((x, skip), dim=1)
            x = decoder(x)

        x = self.head(x)
        return x[..., :height, :width]


model = UNet25D()


## Блок "Обучение модели"

Код и информация о том, как был выстроен пайплайн обучения модели (optimizer/scheduler, loss, metrics, training loop, etc). Графики обучения (train + val) по loss + metrics

In [ ]:
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

EPOCHS = 30
CHECKPOINT_PATH = "best_inpainting_unet.pt"
history = {
    "train_loss": [],
    "val_loss": [],
    "train_r": [],
    "val_r": []
}
best_val_r = -float("inf")


def masked_mse(prediction, target, mask):
    error = (prediction - target) ** 2
    return (error * mask).sum() / mask.sum().clamp_min(1)


def masked_r(prediction, target, mask):
    prediction = prediction[mask.bool()].float()
    target = target[mask.bool()].float()
    prediction = prediction - prediction.mean()
    target = target - target.mean()
    denominator = prediction.norm() * target.norm()
    return ((prediction * target).sum() / denominator.clamp_min(1e-8)).item()


for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    train_r = 0

    for inputs, target, mask in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            prediction = model(inputs)
            loss = masked_mse(prediction, target, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * inputs.size(0)
        train_r += masked_r(prediction.detach(), target, mask)

    train_loss /= len(train_loader.dataset)
    train_r /= len(train_loader)

    model.eval()
    val_loss = 0
    val_r = 0

    with torch.inference_mode():
        for inputs, target, mask in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, enabled=use_amp):
                prediction = model(inputs)
                loss = masked_mse(prediction, target, mask)

            val_loss += loss.item() * inputs.size(0)
            val_r += masked_r(prediction, target, mask)

    val_loss /= len(val_loader.dataset)
    val_r /= len(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_r"].append(train_r)
    history["val_r"].append(val_r)

    scheduler.step(val_r)

    if val_r > best_val_r:
        best_val_r = val_r
        torch.save({
            "model_state_dict": model.state_dict(),
            "mean": mean,
            "std": std
        }, CHECKPOINT_PATH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss: {train_loss:.6f} | val loss: {val_loss:.6f} | "
        f"train R: {train_r:.4f} | val R: {val_r:.4f}"
    )

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_title("Masked MSE")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid()

axes[1].plot(history["train_r"], label="train")
axes[1].plot(history["val_r"], label="validation")
axes[1].set_title("Masked R")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()


## Блок "Тестирование модели"

Код и информация о том, как выглядят результаты предсказания на тестовой выборке

In [ ]:
...

## Блок "Инференса модели"

Код и информация о том, как "затягиваются" пропущенные данные в кубе

In [ ]:
...

## Блок сохранения артефактов

Код и информация о том, как сохранялась модель и восстановленный куб (shape финального куба должен совпадать с shape'ом входного куба, который был скачан по ссылке)

In [ ]:
...